#  EmotionSense — Multi-Face Emotion Recognition Pipeline & MobileNetV2 Training

**Course**: CSE327 Software Engineering  
**Project**: EmotionSense (Facial Emotion Recognition System)  


---

### 📌 20-Step Notebook Master Plan:
1. **Install Dependencies** (`torch`, `torchvision`, `kagglehub`, `mediapipe`, `onnx`, `onnxruntime`, `opencv-python`)
2. **Google Drive Mount & Setup**: Mounts `/content/drive` and automatically persists all model checkpoints & plots to `/content/drive/MyDrive/EmotionSense_Models/`.
3. **Download FER-2013 Dataset** via Kaggle API
4. **Analyze Dataset & Class Imbalance**
5. **Stratified 3-Way Dataset Split**: Train (85%), Validation (15%), and Untouched Test Set (Official FER-2013 Test)
6. **Compute Class Weights Strictly on 85% Train Subset** (Zero validation data leakage)
7. **Data Preprocessing & Augmentations** ($224 \times 224$ RGB, ImageNet normalization, random flips/rotations)
8. **Build MobileNetV2 Architecture** with custom 7-emotion classification head
9. **Stage 1: Classifier Head Warmup** (Entire backbone & BatchNorm stats frozen)
10. **Stage 2 Initialization**: Load `best_stage1.pth` and initialize `best_mobilenetv2.pth` fallback checkpoint
11. **Stage 2: Partial Backbone Fine-Tuning** (`features[-4:]` unfreezing + selective BatchNorm stat updates)
12. **Plot & Save Training & Validation Accuracy/Loss Curves to Google Drive** (`training_curves.png`)
13. **Final Untouched Evaluation on TEST set** (Accuracy, Precision, Recall, F1-Score)
14. **$7 \times 7$ Confusion Matrix Generation & Save to Google Drive** (`confusion_matrix.png`)
15. **Save PyTorch Checkpoint to Google Drive (`best_mobilenetv2.pth`)**
16. **Export to ONNX Engine & Save to Google Drive (`mobilenetv2_emotion.onnx`)**
17. **Verify PyTorch vs. ONNX Prediction Equivalence** across multiple real test set images
18. **Integrate MediaPipe Multi-Face Detection**
19. **Implement Multi-Face Cropping & Batch Inference Engine**
20. **End-to-End Multi-Face Visualization & Domain Gap Report**

---  
## Step 1: Install Dependencies  
**Description**: Installs required PyTorch, Torchvision, Kagglehub, MediaPipe, OpenCV, ONNX, and visualization packages.

In [ ]:
# Step 1: Package Installation
!pip install -q torch torchvision torchaudio --extra-index-url https://download.pytorch.org/whl/cu118
!pip install -q kagglehub mediapipe opencv-python-headless onnx onnxruntime matplotlib seaborn scikit-learn pillow pandas tqdm requests

---  
## Step 2: System Imports, GPU Configuration & Google Drive Integration  
**Description**: Imports standard scientific packages, configures CUDA GPU execution, sets random seeds, and mounts Google Drive (`/content/drive/MyDrive/EmotionSense_Models/`) for automatic output saving.

In [ ]:
# Step 2: Imports, GPU Setup & Google Drive Integration
import os
import sys
import time
import copy
import glob
import random
import shutil
import requests
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
import torchvision
from torchvision import datasets, transforms, models

import mediapipe as mp
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight
import onnx
import onnxruntime

# Set Random Seeds for Full Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# Active Device Setup
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f" Active PyTorch Execution Device: {DEVICE}")
if torch.cuda.is_available():
    print(f" GPU Accelerator: {torch.cuda.get_device_name(0)}")

# Google Drive Mount & Automatic Output Synchronization Setup
IN_COLAB = 'google.colab' in sys.modules
DRIVE_OUTPUT_DIR = './output_models'  # local fallback, used whenever Drive isn't mounted
USING_DRIVE = False

if IN_COLAB:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        DRIVE_OUTPUT_DIR = '/content/drive/MyDrive/EmotionSense_Models'
        os.makedirs(DRIVE_OUTPUT_DIR, exist_ok=True)
        USING_DRIVE = True
        print(f" Google Drive Mounted Successfully! All models & plots will be saved to: {DRIVE_OUTPUT_DIR}")
    except Exception as e:
        #  FIX: previously this branch only printed a notice without creating the
        # local fallback directory, so DRIVE_OUTPUT_DIR pointed at a folder that was
        # never made and every save_to_drive() call silently did nothing all notebook.
        # Now we actually create and use the local fallback when the mount fails.
        os.makedirs(DRIVE_OUTPUT_DIR, exist_ok=True)
        print(f" Google Drive mount failed ({e}). Falling back to local directory: {DRIVE_OUTPUT_DIR}")
else:
    os.makedirs(DRIVE_OUTPUT_DIR, exist_ok=True)
    print(f" Local output directory initialized at: {DRIVE_OUTPUT_DIR}")

def save_to_drive(filename):
    """Persist a local output file into Google Drive (or the local fallback dir if Drive isn't mounted)."""
    if not os.path.exists(filename):
        print(f" save_to_drive skipped: '{filename}' does not exist yet.")
        return
    os.makedirs(DRIVE_OUTPUT_DIR, exist_ok=True)  # safety net in case the dir was removed mid-run
    dest_path = os.path.join(DRIVE_OUTPUT_DIR, os.path.basename(filename))
    shutil.copy(filename, dest_path)
    label = "Google Drive" if USING_DRIVE else "local fallback directory"
    print(f" Saved to {label}: {dest_path}")

---  
## Step 3: Download FER-2013 Dataset  
**Description**: Downloads the official FER-2013 dataset using `kagglehub` and verifies dataset paths.

In [ ]:
# Step 3: Dataset Download
import kagglehub

print(" Downloading FER-2013 dataset from Kaggle...")
dataset_path = kagglehub.dataset_download("msambare/fer2013")
print(f" Dataset downloaded to: {dataset_path}")

OFFICIAL_TRAIN_DIR = os.path.join(dataset_path, "train")
OFFICIAL_TEST_DIR = os.path.join(dataset_path, "test")

EMOTIONS = sorted(os.listdir(OFFICIAL_TRAIN_DIR))
print(f" Identified {len(EMOTIONS)} Emotion Classes: {EMOTIONS}")

---  
## Step 4 & 5: Stratified Split & Class Weight Calculation (Zero Data Leakage)  
**Description**: Uses `train_test_split` with `stratify=targets` to split official training images into **Train (85%)** and **Validation (15%)** while maintaining exact class proportions. Computes class weights **strictly from the 85% train subset**.

In [ ]:
# Step 4 & 5: Stratified Data Split & Clean Class Weight Calculation
full_train_dataset = datasets.ImageFolder(root=OFFICIAL_TRAIN_DIR)
targets = [s[1] for s in full_train_dataset.samples]

# Perform Stratified Train (85%) / Validation (15%) Split
train_indices, val_indices = train_test_split(
    np.arange(len(targets)),
    test_size=0.15,
    stratify=targets,
    random_state=42
)

# Compute Class Weights STRICTLY on 85% Training Subset Labels
train_labels_subset = [targets[i] for i in train_indices]
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(train_labels_subset),
    y=train_labels_subset
)
class_weights_tensor = torch.FloatTensor(class_weights).to(DEVICE)
print(f" Stratified Class Weights (85% Train Subset Only): {dict(zip(EMOTIONS, np.round(class_weights, 3)))}")

---  
## Step 6: Data Preprocessing, Augmentations & DataLoaders  
**Description**: Applies PyTorch augmentations ($224 \times 224$ RGB, random horizontal flip, rotation, color jitter, ImageNet normalization) and instantiates DataLoaders.

In [ ]:
# Step 6: Transforms & DataLoaders
IMAGE_SIZE = 224
BATCH_SIZE = 64

train_transforms = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_test_transforms = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Custom Subset Wrapper for Transforms
class SubsetWithTransform(torch.utils.data.Dataset):
    def __init__(self, subset, transform=None):
        self.subset = subset
        self.transform = transform
    def __getitem__(self, index):
        x, y = self.subset[index]
        if self.transform:
            x = self.transform(x)
        return x, y
    def __len__(self):
        return len(self.subset)

raw_train_subset = Subset(full_train_dataset, train_indices)
raw_val_subset = Subset(full_train_dataset, val_indices)

train_dataset = SubsetWithTransform(raw_train_subset, transform=train_transforms)
val_dataset = SubsetWithTransform(raw_val_subset, transform=val_test_transforms)
test_dataset = datasets.ImageFolder(root=OFFICIAL_TEST_DIR, transform=val_test_transforms)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f" Stratified Split Complete:")
print(f"   - Training Set:   {len(train_dataset)} samples ({len(train_loader)} batches)")
print(f"   - Validation Set: {len(val_dataset)} samples ({len(val_loader)} batches) [Used for Checkpointing & Tuning]")
print(f"   - Untouched Test: {len(test_dataset)} samples ({len(test_loader)} batches) [Used ONLY for Final Report]")

---  
## Step 7: MobileNetV2 Architecture & Selective BatchNorm Protection  
**Description**: Builds MobileNetV2 architecture with custom classifier head `(Dropout(0.3) -> Linear(128) -> ReLU -> Dropout(0.2) -> Linear(7))`. Defines selective `freeze_batchnorm_stats()` ensuring BatchNorm layers in frozen feature blocks stay in `eval()` mode.

In [ ]:
# Step 7: Model Building & Selective BatchNorm Protection
def build_mobilenetv2(num_classes=7, freeze_backbone=True):
    model = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.DEFAULT)

    if freeze_backbone:
        for param in model.features.parameters():
            param.requires_grad = False

    in_features = model.classifier[1].in_features
    model.classifier = nn.Sequential(
        nn.Dropout(0.3),
        nn.Linear(in_features, 128),
        nn.ReLU(),
        nn.Dropout(0.2),
        nn.Linear(128, num_classes)
    )
    return model.to(DEVICE)

model = build_mobilenetv2(num_classes=len(EMOTIONS), freeze_backbone=True)
criterion = nn.CrossEntropyLoss(weight=class_weights_tensor)

# History Dictionary for Plotting Loss & Accuracy Curves
history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}

def freeze_batchnorm_stats(model, unfreeze_top_features=False):
    """
    Freeze BatchNorm running statistics in frozen backbone feature blocks.
    If unfreeze_top_features=True (Stage 2), allows BatchNorm layers inside features[-4:]
    to update stats, while keeping lower frozen backbone BatchNorm layers in eval() mode.
    """
    for module in model.features[:-4].modules():
        if isinstance(module, nn.BatchNorm2d):
            module.eval()

    if unfreeze_top_features:
        for module in model.features[-4:].modules():
            if isinstance(module, nn.BatchNorm2d):
                module.train()
    else:
        for module in model.features[-4:].modules():
            if isinstance(module, nn.BatchNorm2d):
                module.eval()

def train_one_epoch(model, loader, optimizer, criterion, freeze_bn_backbone=True, unfreeze_top_bn=False):
    model.train()
    if freeze_bn_backbone:
        freeze_batchnorm_stats(model, unfreeze_top_features=unfreeze_top_bn)

    total_loss, total_correct, total_samples = 0.0, 0, 0
    for inputs, labels in tqdm(loader, desc="Training", leave=False):
        inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * inputs.size(0)
        _, preds = torch.max(outputs, 1)
        total_correct += torch.sum(preds == labels).item()
        total_samples += inputs.size(0)

    return total_loss / total_samples, total_correct / total_samples

def validate_one_epoch(model, loader, criterion):
    model.eval()
    total_loss, total_correct, total_samples = 0.0, 0, 0
    with torch.no_grad():
        for inputs, labels in tqdm(loader, desc="Validation", leave=False):
            inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
            outputs = model(inputs)
            loss = criterion(outputs, labels)

            total_loss += loss.item() * inputs.size(0)
            _, preds = torch.max(outputs, 1)
            total_correct += torch.sum(preds == labels).item()
            total_samples += inputs.size(0)

    return total_loss / total_samples, total_correct / total_samples

---  
## Step 8: Stage 1 Training — Classifier Head Warmup (10 Epochs)  
**Description**: Trains only the classifier head ($lr=10^{-3}$) with backbone feature parameters and BatchNorm statistics frozen. Records metrics in `history` and saves `best_stage1.pth` locally and to Google Drive.

In [ ]:
# Step 8: Stage 1 Training (Warmup)
optimizer = optim.Adam(model.classifier.parameters(), lr=1e-3)
epochs_stage1 = 10
best_val_acc = 0.0

print(" Starting Stage 1: Classifier Head Warmup (Validation-based Checkpointing)...")
for epoch in range(1, epochs_stage1 + 1):
    train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, criterion, freeze_bn_backbone=True, unfreeze_top_bn=False)
    val_loss, val_acc = validate_one_epoch(model, val_loader, criterion)

    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['train_acc'].append(train_acc)
    history['val_acc'].append(val_acc)

    print(f"Epoch {epoch:02d}/{epochs_stage1:02d} | Train Loss: {train_loss:.4f} Acc: {train_acc*100:.2f}% | Val Loss: {val_loss:.4f} Acc: {val_acc*100:.2f}%")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), 'best_stage1.pth')
        save_to_drive('best_stage1.pth')
        print(f"    Saved Stage 1 Checkpoint based on Validation Acc: {best_val_acc*100:.2f}%")

---  
## Step 9 & 10: Stage 2 Initialization & Partial Backbone Fine-Tuning  
**Description**: Loads `best_stage1.pth` before Stage 2 starts and initializes `best_mobilenetv2.pth` as a fallback safeguard. Unfreezes top MobileNetV2 feature blocks (`features[-4:]`) with selective BatchNorm updates for 20 epochs. Automatically saves checkpoints to Google Drive.

In [ ]:
# Step 9 & 10: Stage 2 Initialization & Partial Backbone Fine-Tuning
#  SAFEGUARD 1: Load best Stage 1 model before starting Stage 2
model.load_state_dict(torch.load('best_stage1.pth', map_location=DEVICE))
print(" Successfully loaded 'best_stage1.pth' before initiating Stage 2.")

#  SAFEGUARD 2: Copy best_stage1.pth to best_mobilenetv2.pth as initial fallback
shutil.copy('best_stage1.pth', 'best_mobilenetv2.pth')
save_to_drive('best_mobilenetv2.pth')
print(" Initialized 'best_mobilenetv2.pth' with Stage 1 checkpoint as fallback safeguard.")

# Unfreeze top feature modules
for param in model.features[-4:].parameters():
    param.requires_grad = True

optimizer = optim.Adam([
    {'params': model.features[-4:].parameters(), 'lr': 1e-4},
    {'params': model.classifier.parameters(), 'lr': 3e-4}
])
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=2)

epochs_stage2 = 20
EARLY_STOPPING_PATIENCE = 6
epochs_without_improvement = 0

print(" Starting Stage 2: Partial Backbone Fine-Tuning...")
for epoch in range(1, epochs_stage2 + 1):
    train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, criterion, freeze_bn_backbone=True, unfreeze_top_bn=True)
    val_loss, val_acc = validate_one_epoch(model, val_loader, criterion)
    scheduler.step(val_acc)

    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['train_acc'].append(train_acc)
    history['val_acc'].append(val_acc)

    print(f"Stage2 Epoch {epoch:02d}/{epochs_stage2:02d} | Train Loss: {train_loss:.4f} Acc: {train_acc*100:.2f}% | Val Loss: {val_loss:.4f} Acc: {val_acc*100:.2f}%")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        epochs_without_improvement = 0
        torch.save(model.state_dict(), 'best_mobilenetv2.pth')
        save_to_drive('best_mobilenetv2.pth')
        print(f"    New Best Stage 2 Checkpoint Saved to Google Drive! Validation Acc: {best_val_acc*100:.2f}%")
    else:
        epochs_without_improvement += 1
        print(f"    No improvement for {epochs_without_improvement}/{EARLY_STOPPING_PATIENCE} epoch(s).")
        if epochs_without_improvement >= EARLY_STOPPING_PATIENCE:
            print(f" Early stopping triggered at epoch {epoch} (no val_acc improvement for {EARLY_STOPPING_PATIENCE} epochs).")
            break

print(f" Final Checkpointing Completed. Best Validation Accuracy Achieved: {best_val_acc*100:.2f}%")

---  
## Step 11: Plot Training & Validation Accuracy / Loss Curves  
**Description**: Plots Loss and Accuracy curves across Stage 1 and Stage 2 epochs and automatically saves `training_curves.png` locally and to Google Drive.

In [ ]:
# Step 11: Plot Training & Validation Learning Curves & Save to Google Drive
plt.figure(figsize=(14, 5))

# Subplot 1: Loss Curve
plt.subplot(1, 2, 1)
plt.plot(history['train_loss'], label='Train Loss', color='blue', linewidth=2)
plt.plot(history['val_loss'], label='Validation Loss', color='red', linestyle='--', linewidth=2)
plt.axvline(x=10, color='gray', linestyle=':', label='Stage 2 Unfreeze Start')
plt.title('MobileNetV2 Training & Validation Loss Curves')
plt.xlabel('Epoch')
plt.ylabel('CrossEntropy Loss')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.6)

# Subplot 2: Accuracy Curve
plt.subplot(1, 2, 2)
plt.plot(np.array(history['train_acc']) * 100, label='Train Acc (%)', color='blue', linewidth=2)
plt.plot(np.array(history['val_acc']) * 100, label='Validation Acc (%)', color='green', linestyle='--', linewidth=2)
plt.axvline(x=10, color='gray', linestyle=':', label='Stage 2 Unfreeze Start')
plt.title('MobileNetV2 Training & Validation Accuracy Curves')
plt.xlabel('Epoch')
plt.ylabel('Accuracy (%)')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.6)

plt.tight_layout()
plt.savefig('training_curves.png', dpi=300)
save_to_drive('training_curves.png')
plt.show()
print(" Training curves saved locally & synced to Google Drive!")

---  
## Step 12 & 13: Final Evaluation on Untouched TEST Set  
**Description**: Loads `best_mobilenetv2.pth` and evaluates performance on the **untouched test set** (`test_loader`), reporting unbiased Accuracy, Precision, Recall, F1-Score, and saving $7 \times 7$ Confusion Matrix (`confusion_matrix.png`) to Google Drive.

In [ ]:
# Step 12 & 13: Untouched Test Set Evaluation & Confusion Matrix Google Drive Sync
model.load_state_dict(torch.load('best_mobilenetv2.pth', map_location=DEVICE))
model.eval()

all_preds, all_labels = [], []
with torch.no_grad():
    for inputs, labels in tqdm(test_loader, desc="Evaluating Untouched Test Set"):
        inputs = inputs.to(DEVICE)
        outputs = model(inputs)
        _, preds = torch.max(outputs, 1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.numpy())

print("\n================ UNTOUCHED TEST SET CLASSIFICATION REPORT ================")
print(classification_report(all_labels, all_preds, target_names=EMOTIONS))

# Confusion Matrix Plot
cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(9, 7))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=EMOTIONS, yticklabels=EMOTIONS)
plt.title('EmotionSense MobileNetV2 — Untouched Test Confusion Matrix')
plt.xlabel('Predicted Emotion')
plt.ylabel('True Emotion')
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=300)
save_to_drive('confusion_matrix.png')
plt.show()
print(" Confusion Matrix saved locally & synced to Google Drive!")

---  
## Step 14, 15 & 16: Export Model to ONNX Engine & Google Drive Sync  
**Description**: Exports model to `mobilenetv2_emotion.onnx`, saves it directly to Google Drive, and strictly verifies that PyTorch prediction outputs match ONNX Runtime predictions across a batch of real test set sample images.

In [ ]:
!pip install -q onnxscript

In [ ]:
# Step 14, 15 & 16: ONNX Export & Google Drive Synchronization
model.eval()
onnx_filename = "mobilenetv2_emotion.onnx"
dummy_input = torch.randn(1, 3, IMAGE_SIZE, IMAGE_SIZE, device=DEVICE)

torch.onnx.export(
    model,
    dummy_input,
    onnx_filename,
    export_params=True,
    opset_version=12,
    do_constant_folding=True,
    input_names=['input'],
    output_names=['output'],
    dynamic_axes={'input': {0: 'batch_size'}, 'output': {0: 'batch_size'}}
)
save_to_drive(onnx_filename)
print(f" Exported & Synced ONNX model engine to Google Drive: {DRIVE_OUTPUT_DIR}/{onnx_filename}")

# Reusable ONNX Runtime session cache
_onnx_session_cache = {}

def get_onnx_session(onnx_model_path):
    """Return a cached onnxruntime.InferenceSession for onnx_model_path, creating it once."""
    if onnx_model_path not in _onnx_session_cache:
        _onnx_session_cache[onnx_model_path] = onnxruntime.InferenceSession(onnx_model_path)
    return _onnx_session_cache[onnx_model_path]

# Verification: Compare PyTorch vs ONNX across a batch of 5 real test images
test_inputs, test_targets = next(iter(test_loader))
sample_batch = test_inputs[0:5].to(DEVICE)

# PyTorch Inference
with torch.no_grad():
    pytorch_logits = model(sample_batch).cpu().numpy()
    pytorch_preds = np.argmax(pytorch_logits, axis=1)

# ONNX Runtime Inference
ort_session = get_onnx_session(onnx_filename)
ort_inputs = {ort_session.get_inputs()[0].name: sample_batch.cpu().numpy()}
onnx_logits = ort_session.run(None, ort_inputs)[0]
onnx_preds = np.argmax(onnx_logits, axis=1)

print(f" PyTorch Batch Predictions: {[EMOTIONS[i] for i in pytorch_preds]}")
print(f" ONNX Batch Predictions:    {[EMOTIONS[i] for i in onnx_preds]}")

# Numerical Equivalence Check
np.testing.assert_allclose(pytorch_logits, onnx_logits, rtol=1e-3, atol=1e-3)
assert np.array_equal(pytorch_preds, onnx_preds), "Mismatch between PyTorch and ONNX predictions!"
print(" VERIFICATION PASSED: PyTorch and ONNX Runtime batch predictions are numerically identical!")

---  
## Step 17 & 18: MediaPipe Multi-Face Detection Integration  
**Description**: Implements Stage A (MediaPipe Face Detection) to extract all face bounding boxes from multi-person images.

In [ ]:
!pip install -q -U mediapipe

In [ ]:
!pip install -q --upgrade "protobuf>=5.28.3,<6"

In [ ]:
import google.protobuf

print("protobuf:", google.protobuf.__version__)

from google.protobuf import runtime_version

print("runtime_version: OK")

import mediapipe as mp

print("MediaPipe:", mp.__version__)
print("MediaPipe import: OK")

In [ ]:
# Step 17 & 18: MediaPipe Tasks Face Detector
# Modern MediaPipe API

import os
import urllib.request
import cv2
import mediapipe as mp

from mediapipe.tasks import python
from mediapipe.tasks.python import vision


# ---------------------------------------------------------
# Download Face Detector model
# ---------------------------------------------------------

FACE_DETECTOR_MODEL = "/content/blaze_face_full_range.tflite"

FACE_DETECTOR_MODEL_URL = (
    "https://storage.googleapis.com/mediapipe-models/"
    "face_detector/blaze_face_full_range/float16/latest/"
    "blaze_face_full_range.tflite"
)

if not os.path.exists(FACE_DETECTOR_MODEL):
    print("Downloading MediaPipe face detector model...")
    urllib.request.urlretrieve(
        FACE_DETECTOR_MODEL_URL,
        FACE_DETECTOR_MODEL
    )
    print("Face detector model downloaded.")
else:
    print("Face detector model already exists.")


# ---------------------------------------------------------
# Face Detector Class
# ---------------------------------------------------------

class MediaPipeFaceDetector:

    def __init__(self, min_detection_confidence=0.5):

        base_options = python.BaseOptions(
            model_asset_path=FACE_DETECTOR_MODEL
        )

        options = vision.FaceDetectorOptions(
            base_options=base_options,
            running_mode=vision.RunningMode.IMAGE,
            min_detection_confidence=min_detection_confidence
        )

        self.detector = vision.FaceDetector.create_from_options(options)

    def detect_faces(self, image_bgr):

        h, w, _ = image_bgr.shape

        # BGR → RGB
        image_rgb = cv2.cvtColor(
            image_bgr,
            cv2.COLOR_BGR2RGB
        )

        # Convert to MediaPipe Image
        mp_image = mp.Image(
            image_format=mp.ImageFormat.SRGB,
            data=image_rgb
        )

        # Detect faces
        detection_result = self.detector.detect(mp_image)

        boxes = []

        for detection in detection_result.detections:

            bbox = detection.bounding_box

            x = int(bbox.origin_x)
            y = int(bbox.origin_y)
            bw = int(bbox.width)
            bh = int(bbox.height)

            # Clamp coordinates
            x = max(0, x)
            y = max(0, y)

            bw = min(bw, w - x)
            bh = min(bh, h - y)

            if bw > 10 and bh > 10:
                boxes.append((x, y, bw, bh))

        return boxes


# ---------------------------------------------------------
# Initialize detector
# ---------------------------------------------------------

face_detector = MediaPipeFaceDetector(
    min_detection_confidence=0.5
)

print(" MediaPipe Tasks Multi-Face Detector initialized!")

In [ ]:
from google.colab import files
import cv2

uploaded = files.upload()

image_path = next(iter(uploaded.keys()))

test_image = cv2.imread(image_path)

test_boxes = face_detector.detect_faces(test_image)

print(f"✅ Detected {len(test_boxes)} face(s).")

for i, box in enumerate(test_boxes, start=1):
    print(f"Face {i}: {box}")

---  
## Step 19: End-to-End Multi-Face Emotion Inference Engine  
**Description**: Combines Stage A (MediaPipe Face Detection) with Stage B (MobileNetV2 ONNX Inference) to classify multiple faces in a single photo/image in batch mode.

In [ ]:
# Step 19: Multi-Face Inference Engine Function
def process_multi_face_emotion(image_path, onnx_model_path="mobilenetv2_emotion.onnx"):
    image_bgr = cv2.imread(image_path)
    if image_bgr is None:
        raise ValueError(f"Could not load image at {image_path}")

    # Stage A: Detect Face Bounding Boxes
    boxes = face_detector.detect_faces(image_bgr)
    print(f" Detected {len(boxes)} face(s) in image.")

    if len(boxes) == 0:
        return image_bgr, []

    # Stage B: Crop & Preprocess Each Face for MobileNetV2
    face_crops = []
    for (x, y, bw, bh) in boxes:
        face_img = image_bgr[y:y+bh, x:x+bw]
        face_rgb = cv2.cvtColor(face_img, cv2.COLOR_BGR2RGB)
        pil_img = Image.fromarray(face_rgb)
        tensor_img = val_test_transforms(pil_img)
        face_crops.append(tensor_img.numpy())

    batch_faces = np.stack(face_crops, axis=0)

    # Stage C: ONNX Batch Inference
    ort_session = get_onnx_session(onnx_model_path)
    ort_inputs = {ort_session.get_inputs()[0].name: batch_faces.astype(np.float32)}
    logits = ort_session.run(None, ort_inputs)[0]
    probs = np.exp(logits) / np.sum(np.exp(logits), axis=1, keepdims=True)

    results = []
    annotated_img = image_bgr.copy()
    for idx, (x, y, bw, bh) in enumerate(boxes):
        top_emotion_idx = np.argmax(probs[idx])
        emotion_label = EMOTIONS[top_emotion_idx]
        confidence = float(probs[idx][top_emotion_idx])

        results.append({
            "face_id": idx + 1,
            "box": {"x": x, "y": y, "w": bw, "h": bh},
            "emotion": emotion_label,
            "confidence": round(confidence, 4)
        })

        # Draw Bounding Box & Label on Image
        cv2.rectangle(annotated_img, (x, y), (x+bw, y+bh), (0, 255, 0), 2)
        label_str = f"{emotion_label}: {confidence*100:.1f}%"
        cv2.putText(annotated_img, label_str, (x, max(y-10, 20)), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)

    return annotated_img, results

print(" Multi-Face Emotion Inference Engine fully configured!")

---  
## 📝 Model Limitations & Domain Shift Discussion Report

### 🔍 Key Technical Insights:
1. **Training Distribution vs. Inference Crop Distribution Shift**:
   - FER-2013 consists of tightly-cropped, grayscale $48 \times 48$ face crops centered closely around eyes, nose, and mouth.
   - During real-world multi-face inference, MediaPipe Face Detector crops bounding boxes from unconstrained, high-resolution color photographs containing background margins, head rotations, and varying lighting.
   - **Impact**: While the pipeline is 100% valid and functional, real-world multi-person inference accuracy may be slightly lower than the test set accuracy achieved on pure FER-2013 images due to this domain gap.

2. **Pipeline Decoupling Benefit**:
   - By decoupling **Face Detection (MediaPipe)** from **Emotion Classification (MobileNetV2)**, the system can scale seamlessly to any number of people in a frame ($N$ faces processed in a single GPU/CPU ONNX batch pass).